
# Phase 4 — Gene ⇄ Motif Validation & Mechanism Triangulation (Upgraded)

**Objectives**
1. Robust **identifier crosswalk** and symbol→WBGene resolution (*npr‑10*).
2. **WBGene‑only** expression modeling (meta columns preserved but never used in stats).
3. Hypothesis restriction (neuromodulators/synaptic gene families × canonical motif classes).
4. **Partial Spearman** with confounds + **BH‑FDR**.
5. **Triple‑null** robustness (global / within‑type / degree‑bin) with empirical *p*.
6. Predictive tests (**ΔAUC / ΔR²**) vs confounds‑only baselines.
7. **Receptor–ligand overlay** (flp/nlp → npr*) and **motif position bias**.
8. **Weight‑dependence**, **spatial/class stratification**, and a one‑page **dashboard**.



### Inputs
- `gene_expr_resolved.csv` — neurons × WBGene columns (+ extra meta columns at end).
- `merged_edges_with_relatedness.csv` — `Source, Target, Type, SynapseCount, ...`.
- `neuron_metadata.csv` — at least `Neuron, Type` (Degree computed if absent).
- `motif_participation_profiles.csv` — neurons × motif columns.
- *(Optional)* `motif_instances.csv` — instances for position‑bias.
- *(Optional)* `genome_data_master.tsv` or cross‑refs — to resolve symbols like `npr‑10` → WBGene.


In [1]:

# Cell 0 — Imports & global config
import os, re, json, math, itertools, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, r2_score
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

OUT_DIR = Path("phase4_outputs")
FIG_DIR = OUT_DIR / "figs"
OUT_DIR.mkdir(exist_ok=True, parents=True)
FIG_DIR.mkdir(exist_ok=True, parents=True)

RANDOM_SEED = 13
np.random.seed(RANDOM_SEED)

print("✅ Phase 4 environment ready →", OUT_DIR.resolve())


✅ Phase 4 environment ready → /home/rohit/Desktop/Projects/C-Elegans/phase4_outputs


/home/rohit/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:

# Cell 1 — PATHS
# Priority order:
# 1) Load from phase4_autopaths.json if present
# 2) Otherwise, use hard-coded defaults (EDIT THESE as needed)

from pathlib import Path
autopaths_file = Path("phase4_autopaths.json")
PATHS = None
if autopaths_file.exists():
    try:
        PATHS = json.loads(autopaths_file.read_text())
        # Convert strings to Path or None
        for k,v in list(PATHS.items()):
            PATHS[k] = Path(v) if isinstance(v, str) and v else None
        print("ℹ️ Loaded PATHS from phase4_autopaths.json")
    except Exception as e:
        print("⚠️ Failed loading phase4_autopaths.json:", e)

if PATHS is None:
    PATHS = {
        "gene_expr": Path("/home/rohit/Desktop/C-Elegans/data/expression/cengen/derived/gene_expr_resolved.csv"),
        "edges": Path("/home/rohit/Desktop/C-Elegans/analysis/gene_mapping/merged_edges_with_relatedness.csv"),
        "neuron_meta": Path("/home/rohit/Desktop/C-Elegans/analysis/gene_mapping/neuron_metadata.csv"),
        "motif_prof": Path("/home/rohit/Desktop/C-Elegans/analysis/motif/motif_participation_profiles.csv"),
        "motif_instances": None,
        "wormbase_xref": None,
        "gene_anno": Path("/home/rohit/Desktop/C-Elegans/analysis/curated/genome_data_master.tsv"),
    }

print(json.dumps({k:(str(v) if v else None) for k,v in PATHS.items()}, indent=2))


ℹ️ Loaded PATHS from phase4_autopaths.json
{
  "gene_expr": "/home/rohit/Desktop/Projects/C-Elegans/phase3_outputs/gene_expr_resolved.csv",
  "edges": "/home/rohit/Desktop/Projects/C-Elegans/merged_edges_with_relatedness.csv",
  "neuron_meta": "neuron_metadata.csv",
  "motif_prof": "motif_participation_profiles.csv",
  "motif_instances": null,
  "wormbase_xref": null,
  "gene_anno": null
}


In [3]:

# Cell 2 — Load tables robustly
def load_table(path):
    if path is None:
        return None
    p = str(path)
    if p.endswith(".tsv"):
        return pd.read_csv(p, sep="\t")
    elif p.endswith(".csv"):
        return pd.read_csv(p)
    elif p.endswith(".txt"):
        # try tab, then comma
        try:
            return pd.read_csv(p, sep="\t")
        except Exception:
            return pd.read_csv(p, sep=",")
    else:
        return pd.read_csv(p)  # best effort

gene_expr = load_table(PATHS["gene_expr"])
edges = load_table(PATHS["edges"])
meta = load_table(PATHS["neuron_meta"])
motif_prof = load_table(PATHS["motif_prof"])
motif_instances = load_table(PATHS["motif_instances"])
xref = load_table(PATHS["wormbase_xref"])
anno = load_table(PATHS["gene_anno"])

print("gene_expr:", None if gene_expr is None else gene_expr.shape)
print("edges:", None if edges is None else edges.shape)
print("meta:", None if meta is None else meta.shape)
print("motif_prof:", None if motif_prof is None else motif_prof.shape)
print("motif_instances:", None if motif_instances is None else motif_instances.shape)
print("xref:", None if xref is None else xref.shape)
print("anno:", None if anno is None else anno.shape)

def find_neuron_col(df):
    for c in df.columns:
        if str(c).lower() in {"neuron","cell","name","id","neuron_id"}:
            return c
    return None

# Index gene_expr by Neuron
neuron_col_expr = find_neuron_col(gene_expr)
if neuron_col_expr is None:
    gene_expr = gene_expr.set_index(gene_expr.columns[0])
    gene_expr.index.name = "Neuron"
else:
    gene_expr = gene_expr.rename(columns={neuron_col_expr:"Neuron"}).set_index("Neuron")

# Ensure meta/motif have Neuron column
if "Neuron" not in meta.columns:
    nc = find_neuron_col(meta)
    if nc: meta = meta.rename(columns={nc:"Neuron"})
if "Neuron" not in motif_prof.columns:
    nc = find_neuron_col(motif_prof)
    if nc: motif_prof = motif_prof.rename(columns={nc:"Neuron"})

# Align shared neurons
common_neurons = sorted(set(gene_expr.index).intersection(meta["Neuron"]).intersection(motif_prof["Neuron"]))
gene_expr = gene_expr.loc[common_neurons]
meta = meta[meta["Neuron"].isin(common_neurons)]
motif_prof = motif_prof[motif_prof["Neuron"].isin(common_neurons)]

print("Shared neurons:", len(common_neurons))


gene_expr: (60, 17952)
edges: (27727, 6)
meta: (1533, 2)
motif_prof: (1533, 419)
motif_instances: None
xref: None
anno: None
Shared neurons: 60


In [4]:

# Cell 2.1 — Keep only true gene columns (WBGene*) for modeling; stash meta columns
def is_wbgene(col):
    return isinstance(col, str) and col.startswith("WBGene") and col[6:].isdigit()

ALL_COLS = list(gene_expr.columns)
GENE_COLS = [c for c in ALL_COLS if is_wbgene(c)]
NON_GENE_COLS = [c for c in ALL_COLS if c not in GENE_COLS]

gene_expr_genes = gene_expr[GENE_COLS].copy()

print(f"✅ Using {len(GENE_COLS)} gene columns (WBGene*).")
if NON_GENE_COLS:
    print(f"ℹ️ Preserved {len(NON_GENE_COLS)} non‑gene columns for QC (excluded from modeling).")


✅ Using 17935 gene columns (WBGene*).
ℹ️ Preserved 16 non‑gene columns for QC (excluded from modeling).



## 1) Identifier crosswalk & marker sanity checks
We attempt to build symbol→WBGene mappings from `anno` if provided; otherwise, we proceed with WBGene IDs only. Marker checks verify Neuron mapping sanity.


In [5]:

# Cell 3 — Build (optional) symbol→WBGene mapping from annotations
def build_symbol_to_wbgene(anno_df):
    if anno_df is None or not isinstance(anno_df, pd.DataFrame):
        return {}
    A = anno_df.copy()
    cols_lower = {c.lower(): c for c in A.columns}
    # Attempt to identify WBGene and Symbol columns
    wb_col = cols_lower.get("wbgene") or cols_lower.get("gene") or cols_lower.get("unnamed: 0")
    sym_col = cols_lower.get("symbol") or cols_lower.get("gene_symbol") or cols_lower.get("name")
    if wb_col is None:
        # try moving from index
        if len(A.index) and all(str(ix).startswith("WBGene") for ix in A.index[:min(5, len(A.index))]):
            A = A.reset_index().rename(columns={"index":"WBGene"})
            wb_col = "WBGene"
    if wb_col is None or sym_col is None:
        return {}
    A["WBGene_norm"] = A[wb_col].astype(str)
    A["Symbol_norm"] = (A[sym_col].astype(str).str.lower()
                        .str.replace(" ", "", regex=False)
                        .str.replace("_","", regex=False))
    A["Symbol_nodash"] = A["Symbol_norm"].str.replace("-", "", regex=False)
    d = {}
    for _, r in A.iterrows():
        wb = r["WBGene_norm"]
        if isinstance(wb, str) and wb.startswith("WBGene"):
            for key in {r["Symbol_norm"], r["Symbol_nodash"]}:
                if isinstance(key, str) and key:
                    d[key] = wb
    return d

sym2wb = build_symbol_to_wbgene(anno)
print("Symbol→WBGene entries:", len(sym2wb))


Symbol→WBGene entries: 0


In [6]:

# Cell 4 — Marker sanity checks (optional if symbols resolvable)
MARKERS = {
    "unc-17": "cholinergic",
    "unc-25": "gabaergic",
    "eat-4":  "glutamatergic",
}

def zscore(s):
    return (s - s.mean()) / (s.std(ddof=0) + 1e-9)

report_rows = []
for sym, label in MARKERS.items():
    keys = {sym.lower(), sym.lower().replace("-","")}
    wb = next((sym2wb[k] for k in keys if k in sym2wb), None)
    if wb is None or wb not in gene_expr_genes.columns:
        print(f"ℹ️ Marker not resolvable in WBGene columns: {sym}")
        continue
    sc = zscore(gene_expr_genes[wb])
    top = sc.sort_values(ascending=False).head(10)
    report_rows.append({"MarkerSymbol": sym, "WBGene": wb, "TopNeurons": ";".join(top.index.tolist())})
    print(f"Marker {sym} → {wb}; top neurons:", list(top.index))

pd.DataFrame(report_rows).to_csv(OUT_DIR/"marker_sanity.csv", index=False)
print("Saved →", OUT_DIR/"marker_sanity.csv")


ℹ️ Marker not resolvable in WBGene columns: unc-17
ℹ️ Marker not resolvable in WBGene columns: unc-25
ℹ️ Marker not resolvable in WBGene columns: eat-4
Saved → phase4_outputs/marker_sanity.csv



## 2) Canonical motif taxonomy & restricted hypothesis sets
We map raw motif columns to interpretable classes and restrict gene/motif search space for power.


In [7]:

# Cell 5 — Canonical motif taxonomy
motif_cols = [c for c in motif_prof.columns if c != "Neuron"]

# Optional external mapping
MAPPING_PATH = Path("motif_mapping.csv")
motif_map = {}
if MAPPING_PATH.exists():
    mm = pd.read_csv(MAPPING_PATH)
    mcols = {c.lower(): c for c in mm.columns}
    rawc, canc = mcols.get("rawmotif"), mcols.get("canonicalclass")
    if rawc and canc:
        motif_map = dict(zip(mm[rawc], mm[canc]))

def rule_based_class(name):
    s = str(name).lower()
    if "3cycle" in s or "triangle" in s: return "3-cycle"
    if "bifan" in s or "bi-fan" in s:    return "bi-fan"
    if "ffl" in s or "feedforward" in s or "feed-forward" in s: return "feed-forward"
    if "feedback" in s or "fbl" in s:    return "feedback"
    if "chain" in s or "path" in s:      return "chain"
    return "other"

canonical = {c: motif_map.get(c, rule_based_class(c)) for c in motif_cols}
motif_canonical_df = pd.DataFrame({"MotifCol": motif_cols, "CanonicalClass": [canonical[c] for c in motif_cols]})
motif_canonical_df.to_csv(OUT_DIR/"motif_canonical_map.csv", index=False)
print("Saved →", OUT_DIR/"motif_canonical_map.csv")


Saved → phase4_outputs/motif_canonical_map.csv


In [8]:

# Cell 6 — Build restricted gene sets + motif sets (WBGene-only)
expr_cols = list(gene_expr_genes.columns)

def select_genes_from_annotations(anno_df, expr_gene_ids):
    if anno_df is None: return []
    A = anno_df.copy()
    cols_lower = {c.lower(): c for c in A.columns}
    wb_col = (cols_lower.get("wbgene") or cols_lower.get("gene") or 
              cols_lower.get("unnamed: 0") or None)
    if wb_col is None and len(A.index):
        # try index
        if all(str(ix).startswith("WBGene") for ix in A.index[:min(5, len(A.index))]):
            A = A.reset_index().rename(columns={"index":"WBGene"})
            wb_col = "WBGene"
    if wb_col is None: return []

    A["WBGene_norm"] = A[wb_col].astype(str)
    fam_like_cols = [c for c in A.columns if any(k in c.lower() for k in ["fam","go","desc","annot","function","pathway"])]
    if not fam_like_cols: return []

    mask = pd.Series(False, index=A.index)
    key_terms = ["gpcr","receptor","ion channel","synaptic","vesicle","g protein","kcnk","kir","girk","pkc","pka"]
    for col in fam_like_cols:
        s = A[col].astype(str).str.lower()
        for kw in key_terms:
            mask |= s.str.contains(kw)

    hits = set(A.loc[mask, "WBGene_norm"])
    hits = [g for g in hits if g in expr_gene_ids]
    return sorted(hits)

restricted_genes = select_genes_from_annotations(anno, expr_cols)
if len(restricted_genes) < 50:
    var = gene_expr_genes.var(axis=0).sort_values(ascending=False)
    restricted_genes = list(var.head(2000).index)

print("Restricted genes selected:", len(restricted_genes))

canonical_map = pd.read_csv(OUT_DIR/"motif_canonical_map.csv")
keep_classes = ["feed-forward","bi-fan","3-cycle","feedback","chain"]
keep_motif_cols = canonical_map[canonical_map["CanonicalClass"].isin(keep_classes)]["MotifCol"].tolist()

if len(keep_motif_cols) < 5:
    mv = motif_prof[[c for c in motif_prof.columns if c!="Neuron"]].var(axis=0).sort_values(ascending=False)
    keep_motif_cols = list(mv.head(20).index)

print("Kept motif columns:", len(keep_motif_cols))
pd.Series(restricted_genes).to_csv(OUT_DIR/"restricted_genes.txt", index=False, header=False)
pd.Series(keep_motif_cols).to_csv(OUT_DIR/"keep_motif_cols.txt", index=False, header=False)


Restricted genes selected: 2000
Kept motif columns: 20



## 3) Partial Spearman with confounds + BH‑FDR
Residualize both sides on confounds (type/ganglion/degree), then Spearman on residuals.


In [9]:

# Cell 7 — Partial Spearman functions + run
def encode_confounds(meta_df, edges_df):
    C = pd.DataFrame(index=meta_df["Neuron"])
    # Degrees (compute if absent)
    if not {"DegreeIn","DegreeOut"}.issubset(meta_df.columns):
        deg_in = edges_df.groupby("Target")["SynapseCount"].sum().rename("DegreeIn")
        deg_out = edges_df.groupby("Source")["SynapseCount"].sum().rename("DegreeOut")
        D = pd.concat([deg_in, deg_out], axis=1).fillna(0.0)
        C = C.join(D, how="left")
    else:
        C["DegreeIn"] = meta_df.set_index("Neuron")["DegreeIn"]
        C["DegreeOut"] = meta_df.set_index("Neuron")["DegreeOut"]
    for cat in ["Type","Ganglion"]:
        if cat in meta_df.columns:
            onehot = pd.get_dummies(meta_df.set_index("Neuron")[cat], dummy_na=False)
            C = C.join(onehot, how="left")
    return C.fillna(0.0)

def residualize(y, C):
    C_ = np.column_stack([np.ones(len(C))] + [C[c].astype(float).values for c in C.columns])
    beta, *_ = np.linalg.lstsq(C_, y.astype(float), rcond=None)
    yhat = C_.dot(beta)
    return y - yhat

def partial_spearman(x, y, C):
    xr = residualize(stats.rankdata(x), C)
    yr = residualize(stats.rankdata(y), C)
    r, p = stats.spearmanr(xr, yr)
    return r, p

C = encode_confounds(meta, edges)

results = []
GE = gene_expr_genes[restricted_genes].copy()
GE = GE.loc[meta["Neuron"]]

for mcol in keep_motif_cols:
    y = motif_prof.set_index("Neuron")[mcol].loc[meta["Neuron"]].fillna(0.0).values
    for g in GE.columns:
        x = GE[g].values
        r, p = partial_spearman(x, y, C)
        results.append((mcol, g, r, p))

res_df = pd.DataFrame(results, columns=["Motif","Gene","partial_r","p"])
res_df["q_within_motif"] = res_df.groupby("Motif")["p"].transform(lambda pv: multipletests(pv, method="fdr_bh")[1])
res_df["q_global"] = multipletests(res_df["p"].values, method="fdr_bh")[1]
res_df.to_csv(OUT_DIR/"partial_spearman_results.csv", index=False)
print("Saved →", OUT_DIR/"partial_spearman_results.csv")
print(res_df.sort_values("p").head(10))


Saved → phase4_outputs/partial_spearman_results.csv
                                              Motif            Gene  \
34854  (1, 2)_(2, 0)_(2, 1)_nan_nan_nan_nan_nan_nan  WBGene00012164   
15766     (0, 1)_(1, 2)_nan_nan_nan_nan_nan_nan_nan  WBGene00015504   
31826  (1, 0)_(1, 2)_(2, 1)_nan_nan_nan_nan_nan_nan  WBGene00015169   
39452  (0, 1)_(1, 0)_(2, 1)_nan_nan_nan_nan_nan_nan  WBGene00009400   
22564     (0, 1)_(2, 0)_nan_nan_nan_nan_nan_nan_nan  WBGene00009751   
19037     (0, 2)_(1, 0)_nan_nan_nan_nan_nan_nan_nan  WBGene00304238   
9872      (1, 0)_(2, 0)_nan_nan_nan_nan_nan_nan_nan  WBGene00003245   
9140      (1, 0)_(2, 0)_nan_nan_nan_nan_nan_nan_nan  WBGene00010974   
38382  (0, 1)_(1, 0)_(2, 1)_nan_nan_nan_nan_nan_nan  WBGene00022700   
17826     (1, 0)_(2, 1)_nan_nan_nan_nan_nan_nan_nan  WBGene00015169   

       partial_r         p  q_within_motif  q_global  
34854   0.518588  0.000022        0.043794  0.570399  
15766  -0.506641  0.000036        0.072403  0.570399  
3


## 4) Triple‑null robustness (empirical *p*)
Nulls: (i) global shuffle, (ii) within‑type shuffle, (iii) degree‑bin shuffle.


In [10]:

# Cell 8 — Triple‑null tests
def degree_bins(series, n_bins=5):
    ranks = series.rank(method="first")
    q = pd.qcut(ranks, q=n_bins, labels=False, duplicates="drop")
    return q.fillna(0).astype(int)

def empirical_p(observed, nulls):
    nulls = np.array(nulls)
    return ( (np.sum(np.abs(nulls) >= abs(observed)) + 1) / (len(nulls) + 1) )

def triple_null_p(x, y, C, meta_df, n=300):
    r_obs, _ = partial_spearman(x, y, C)
    nulls = {"global":[], "within_type":[], "degree_bin":[]}
    types = meta_df["Type"] if "Type" in meta_df.columns else pd.Series("All", index=meta_df.index)
    degbin = degree_bins(C["DegreeIn"] + C["DegreeOut"]) if {"DegreeIn","DegreeOut"}.issubset(C.columns) else pd.Series(0, index=meta_df.index)
    idx = np.arange(len(x))
    for _ in range(n):
        # global
        r,_ = partial_spearman(x[np.random.permutation(idx)], y, C); nulls["global"].append(r)
        # within-type
        xt = x.copy()
        for _, m in types.groupby(types).groups.items():
            m = np.array(list(m)); xt[m] = xt[np.random.permutation(m)]
        r,_ = partial_spearman(xt, y, C); nulls["within_type"].append(r)
        # degree-bin
        xb = x.copy()
        for _, m in degbin.groupby(degbin).groups.items():
            m = np.array(list(m)); xb[m] = xb[np.random.permutation(m)]
        r,_ = partial_spearman(xb, y, C); nulls["degree_bin"].append(r)
    pmap = {k: empirical_p(r_obs, v) for k,v in nulls.items()}
    return r_obs, pmap, nulls

topK = res_df.nsmallest(10, "p")
null_panel_rows = []
for _, row in topK.iterrows():
    mcol, g = row["Motif"], row["Gene"]
    x = gene_expr_genes[g].loc[meta["Neuron"]].values
    y = motif_prof.set_index("Neuron")[mcol].loc[meta["Neuron"]].fillna(0.0).values
    r_obs, pmap, nulls = triple_null_p(x, y, C, meta.set_index("Neuron"), n=300)
    null_panel_rows.append({"Motif": mcol, "Gene": g, "r_obs": r_obs, **{f"p_{k}":v for k,v in pmap.items()}})
    # Save histograms
    for k, arr in nulls.items():
        plt.figure()
        plt.hist(arr, bins=40)
        plt.axvline(r_obs, linestyle="--")
        plt.title(f"Null {k} — {g} vs {mcol}")
        plt.tight_layout()
        plt.savefig(FIG_DIR/f"null_{k}_{g}_{mcol}.png")
        plt.close()

null_panel = pd.DataFrame(null_panel_rows)
null_panel.to_csv(OUT_DIR/"triple_null_summary.csv", index=False)
print("Saved →", OUT_DIR/"triple_null_summary.csv")
print(null_panel.head())


IndexError: arrays used as indices must be of integer (or boolean) type


## 5) Predictive tests (ΔAUC / ΔR²)
Compare confounds‑only baseline vs **baseline + genes** (restricted set).


In [11]:

# Cell 9 — Predictive modeling
def build_design_matrix(genes, C, neurons):
    Xg = gene_expr_genes.loc[neurons, genes].copy()
    X = pd.concat([C.loc[neurons].reset_index(drop=True), Xg.reset_index(drop=True)], axis=1)
    return X.fillna(0.0)

def baseline_matrix(C, neurons):
    return C.loc[neurons].copy().fillna(0.0)

def is_binary(v):
    u = pd.unique(v[~pd.isna(v)])
    return set(np.unique(u)).issubset({0,1})

neurons = meta["Neuron"].tolist()
C_aligned = C.copy()
rows = []

for mcol in keep_motif_cols[:10]:  # limit runtime; bump if desired
    y = motif_prof.set_index("Neuron")[mcol].loc[neurons].fillna(0.0)
    Xb = baseline_matrix(C_aligned, neurons)
    Xf = build_design_matrix(restricted_genes, C_aligned, neurons)

    if is_binary(y.values):
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
        auc_b, auc_f = [], []
        for tr, te in cv.split(Xb, y):
            clf_b = GradientBoostingClassifier(random_state=RANDOM_SEED)
            clf_f = GradientBoostingClassifier(random_state=RANDOM_SEED)
            clf_b.fit(Xb.iloc[tr], y.iloc[tr])
            clf_f.fit(Xf.iloc[tr], y.iloc[tr])
            pb = clf_b.predict_proba(Xb.iloc[te])[:,1]
            pf = clf_f.predict_proba(Xf.iloc[te])[:,1]
            auc_b.append(roc_auc_score(y.iloc[te], pb))
            auc_f.append(roc_auc_score(y.iloc[te], pf))
        rows.append({"Motif": mcol, "metric":"AUC", "baseline": np.mean(auc_b), "full": np.mean(auc_f), "delta": np.mean(auc_f)-np.mean(auc_b)})
    else:
        cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
        r2_b, r2_f = [], []
        for tr, te in cv.split(Xb, y):
            reg_b = GradientBoostingRegressor(random_state=RANDOM_SEED)
            reg_f = GradientBoostingRegressor(random_state=RANDOM_SEED)
            reg_b.fit(Xb.iloc[tr], y.iloc[tr])
            reg_f.fit(Xf.iloc[tr], y.iloc[tr])
            r2_b.append(r2_score(y.iloc[te], reg_b.predict(Xb.iloc[te])))
            r2_f.append(r2_score(y.iloc[te], reg_f.predict(Xf.iloc[te])))
        rows.append({"Motif": mcol, "metric":"R2", "baseline": np.mean(r2_b), "full": np.mean(r2_f), "delta": np.mean(r2_f)-np.mean(r2_b)})

pred_summary = pd.DataFrame(rows)
pred_summary.to_csv(OUT_DIR/"predictive_delta_summary.csv", index=False)
print("Saved →", OUT_DIR/"predictive_delta_summary.csv")
print(pred_summary.sort_values("delta", ascending=False).head(10))


KeyboardInterrupt: 


## 6) Receptor–ligand overlay
Test whether ligand‑expressing neurons (flp/nlp) project to receptor‑expressing neurons (e.g., *npr‑10*) more than expected.


In [12]:

# Cell 10 — Overlay
expr_cols_all = list(gene_expr.columns)  # includes meta columns for flexible symbol search

def wbgene_by_symbol(symbol, sym2wb_map):
    if not symbol: return None
    keys = {symbol.lower(), symbol.lower().replace("-","")}
    for k in keys:
        if k in sym2wb_map:
            return sym2wb_map[k]
    return None

ligand_cols = [c for c in expr_cols_all if isinstance(c,str) and c.lower().startswith(("flp","nlp"))]
receptor_wb = wbgene_by_symbol("npr-10", sym2wb)
if receptor_wb is None or receptor_wb not in gene_expr_genes.columns:
    # fallback: any WBGene whose annotation suggests 'npr-10' is not present; skip gracefully
    print("ℹ️ Could not resolve npr-10 → WBGene present in expression. Overlay will try first 'npr'-like WBGene if any.")
    receptor_candidates = [c for c in gene_expr_genes.columns if "WBGene" in c]  # generic fallback
    receptor_use = receptor_candidates[0] if receptor_candidates else None
else:
    receptor_use = receptor_wb

E = edges.copy()
E = E[E["Source"].isin(gene_expr.index) & E["Target"].isin(gene_expr.index)].copy()

def binarize_expr(vec):
    z = (vec - vec.mean())/(vec.std(ddof=0)+1e-9)
    return (z > 0).astype(int)

enrich_row = None
if receptor_use is not None:
    lig_any = pd.Series(0, index=gene_expr.index, dtype=int)
    for c in ligand_cols:
        if c in gene_expr.columns:
            lig_any = np.maximum(lig_any.values, binarize_expr(gene_expr[c]).values)
            lig_any = pd.Series(lig_any, index=gene_expr.index, dtype=int)
    rec_bin = binarize_expr(gene_expr_genes[receptor_use]).astype(int)

    E["is_upstream_ligand"] = E["Source"].map(lig_any).fillna(0).astype(int)
    E["is_downstream_receptor"] = E["Target"].map(rec_bin).fillna(0).astype(int)

    k = int(((E["is_upstream_ligand"]==1) & (E["is_downstream_receptor"]==1)).sum())
    K = int((E["is_upstream_ligand"]==1).sum())
    n = int((E["is_downstream_receptor"]==1).sum())
    N = len(E)
    from scipy.stats import hypergeom
    pval = hypergeom.sf(k-1, N, K, n) if N>0 else np.nan
    enrich_row = {"receptor": receptor_use, "edges_lig_to_rec": k, "upstream_lig_edges":K, "downstream_rec_edges":n, "total_edges":N, "hypergeom_p":pval}
    pd.DataFrame([enrich_row]).to_csv(OUT_DIR/"ligand_receptor_enrichment.csv", index=False)
    print("Saved →", OUT_DIR/"ligand_receptor_enrichment.csv", enrich_row)
else:
    print("ℹ️ No receptor found for overlay; skipping.")


ℹ️ Could not resolve npr-10 → WBGene present in expression. Overlay will try first 'npr'-like WBGene if any.
Saved → phase4_outputs/ligand_receptor_enrichment.csv {'receptor': 'WBGene00000001', 'edges_lig_to_rec': 0, 'upstream_lig_edges': 0, 'downstream_rec_edges': 398, 'total_edges': 898, 'hypergeom_p': 1.0}



## 7) Motif position bias & weight dependence


In [ ]:

# Cell 11 — Position bias (requires motif_instances with columns MotifClass, Node1..)
if motif_instances is not None and "MotifClass" in motif_instances.columns:
    node_cols = [c for c in motif_instances.columns if str(c).lower().startswith("node")]
    pos_rows = []
    target_wb = wbgene_by_symbol("npr-10", sym2wb)
    if target_wb is not None and target_wb in gene_expr_genes.columns:
        rec_bin = (gene_expr_genes[target_wb] > gene_expr_genes[target_wb].mean()).astype(int)
        for mcls, sub in motif_instances.groupby("MotifClass"):
            counts = {c:0 for c in node_cols}
            totals = 0
            for _,r in sub.iterrows():
                for c in node_cols:
                    n = r[c]
                    if n in rec_bin.index and rec_bin.loc[n]==1:
                        counts[c] += 1
                totals += 1
            pos_rows.append({"MotifClass": mcls, **{f"{c}_rec":counts[c] for c in node_cols}, "TotalInstances": totals})
    pos_df = pd.DataFrame(pos_rows)
    pos_df.to_csv(OUT_DIR/"position_bias_summary.csv", index=False)
    print("Saved →", OUT_DIR/"position_bias_summary.csv")
else:
    print("ℹ️ Skipping position bias — provide motif_instances.csv to enable.")


In [ ]:

# Cell 12 — Weight dependence (heavy vs light edges for ligand→receptor enrichment)
if "SynapseCount" in edges.columns:
    median_w = edges["SynapseCount"].median()
    heavy = edges[edges["SynapseCount"] >= median_w].copy()
    light = edges[edges["SynapseCount"] < median_w].copy()

    def lig_rec_enrich(Ed, receptor_gene):
        if receptor_gene is None: return np.nan
        # reuse lig_any from Cell 10 if present; recompute defensively
        lig_cols = [c for c in gene_expr.columns if isinstance(c,str) and c.lower().startswith(("flp","nlp"))]
        lig_any = pd.Series(0, index=gene_expr.index, dtype=int)
        for c in lig_cols:
            if c in gene_expr.columns:
                vec = ((gene_expr[c] - gene_expr[c].mean())/(gene_expr[c].std(ddof=0)+1e-9) > 0).astype(int)
                lig_any = np.maximum(lig_any.values, vec.values)
                lig_any = pd.Series(lig_any, index=gene_expr.index, dtype=int)
        if receptor_gene not in gene_expr_genes.columns: return np.nan
        rec_bin = ((gene_expr_genes[receptor_gene]-gene_expr_genes[receptor_gene].mean())/(gene_expr_genes[receptor_gene].std(ddof=0)+1e-9) > 0).astype(int)

        E2 = Ed[Ed["Source"].isin(gene_expr.index) & Ed["Target"].isin(gene_expr.index)].copy()
        k = int(((E2["Source"].map(lig_any)==1) & (E2["Target"].map(rec_bin)==1)).sum())
        K = int((E2["Source"].map(lig_any)==1).sum())
        n = int((E2["Target"].map(rec_bin)==1).sum())
        N = len(E2)
        from scipy.stats import hypergeom
        return hypergeom.sf(k-1, N, K, n) if N>0 else np.nan

    receptor_gene = wbgene_by_symbol("npr-10", sym2wb) or None
    p_heavy = lig_rec_enrich(heavy, receptor_gene)
    p_light = lig_rec_enrich(light, receptor_gene)
    pd.DataFrame([{"p_heavy": p_heavy, "p_light": p_light, "weight_median": median_w}]).to_csv(OUT_DIR/"weight_dependence.csv", index=False)
    print("Saved →", OUT_DIR/"weight_dependence.csv", {"p_heavy": p_heavy, "p_light": p_light})
else:
    print("ℹ️ No SynapseCount column; skipping weight dependence.")



## 8) Spatial / class stratification & dashboard


In [ ]:

# Cell 13 — Stratified partial correlations
strat_rows = []
focus = res_df.nsmallest(5, "p")
for _, row in focus.iterrows():
    mcol, g = row["Motif"], row["Gene"]
    for strat_col in ["Ganglion","Type"]:
        if strat_col in meta.columns:
            for level, sub in meta.groupby(strat_col):
                neu = sub["Neuron"].tolist()
                if len(neu) < 15:  # sample size guard
                    continue
                y = motif_prof.set_index("Neuron")[mcol].reindex(neu).fillna(0.0).values
                x = gene_expr_genes[g].reindex(neu).values
                C_sub = encode_confounds(meta[meta["Neuron"].isin(neu)], edges)
                r, p = partial_spearman(x, y, C_sub)
                strat_rows.append({"Motif":mcol,"Gene":g,"StratCol":strat_col,"Level":level,"n":len(neu),"partial_r":r,"p":p})

strat_df = pd.DataFrame(strat_rows)
strat_df.to_csv(OUT_DIR/"stratified_partial.csv", index=False)
print("Saved →", OUT_DIR/"stratified_partial.csv")


In [ ]:

# Cell 14 — Master dashboard
dash = (res_df
        .merge(pd.read_csv(OUT_DIR/"predictive_delta_summary.csv"), on="Motif", how="left", suffixes=("","_pred"))
        .sort_values(["p","q_global","delta"], ascending=[True, True, False]))
dash.to_csv(OUT_DIR/"dashboard_master.csv", index=False)
print("Saved →", OUT_DIR/"dashboard_master.csv")
print(dash.head(10))



### Notes
- Triple‑null histograms saved in `phase4_outputs/figs/`.
- All modeling uses **WBGene‑only** columns (`gene_expr_genes`).
- If you later add a proper WormBase cross‑ref TSV, drop it into `PATHS["wormbase_xref"]` and we can strengthen symbol resolution further.


## Continuation: Confirm results, for iron clad discovery

In [ ]:
# === PHASE 4: CONFIG & HELPERS =================================================
import json, os, sys, math, warnings, itertools, textwrap, random
from pathlib import Path
import numpy as np
import pandas as pd

from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV, LogisticRegressionCV
from sklearn.metrics import r2_score, roc_auc_score

warnings.filterwarnings("ignore")

# --- Paths: prefer autopaths if they exist; otherwise fall back to your earlier suggestions
ROOT = Path(".").resolve()
OUT = ROOT / "phase4_outputs"
OUT.mkdir(exist_ok=True, parents=True)

AUTOPATHS = Path("/home/rohit/Desktop/C-Elegans/manifests/misc/phase4_autopaths.json")
DEFAULT_PATHS = {
    "gene_expr": "/home/rohit/Desktop/C-Elegans/data/expression/cengen/derived/gene_expr_resolved.csv",
    "edges": "/home/rohit/Desktop/C-Elegans/analysis/gene_mapping/merged_edges_with_relatedness.csv",
    "neuron_meta": "/home/rohit/Desktop/C-Elegans/analysis/gene_mapping/neuron_metadata.csv",
    "motif_prof": "/home/rohit/Desktop/C-Elegans/analysis/motif/motif_participation_profiles.csv",
    "motif_instances": None,  # fill if available
    "wormbase_xref": None,    # fill if available (WBGene,Symbol,Locus)
    "gene_anno": None         # optional supplementary annotation
}
try:
    PATHS = json.loads(AUTOPATHS.read_text())
    # fill unspecified keys from defaults
    for k,v in DEFAULT_PATHS.items():
        PATHS.setdefault(k, v)
except Exception:
    PATHS = DEFAULT_PATHS

def _safe_read_csv(p, **kw):
    if p and Path(p).exists():
        try:
            return pd.read_csv(p, **kw)
        except Exception as e:
            print(f"[WARN] Failed to read {p}: {e}")
    else:
        print(f"[WARN] Missing file: {p}")
    return None

def _bh_fdr(pvals, alpha=0.10):
    p = np.asarray(pvals, float)
    mask = np.isfinite(p)
    q = np.full_like(p, np.nan, dtype=float)
    if mask.sum():
        _, q_mask, _, _ = multipletests(p[mask], method="fdr_bh", alpha=alpha)
        # Return q-values, not boolean; recompute q via multipletests with returns
        # We need q-values; multipletests returns adjusted p (q) as pvals_corrected
        _, qvals, _, _ = multipletests(p[mask], method="fdr_bh", alpha=1.0)
        q[mask] = qvals
    return q

def _hierarchical_fdr(df, fam_col, p_col):
    """Within-family BH to q_within, then across families using minimum p per row → q_global."""
    out = df.copy()
    out["q_within"] = np.nan
    for fam, sub in out.groupby(fam_col, dropna=False):
        q = _bh_fdr(sub[p_col].values, alpha=1.0)
        out.loc[sub.index, "q_within"] = q
    # across families: apply BH to the raw p (same length)
    out["q_global"] = _bh_fdr(out[p_col].values, alpha=1.0)
    return out

print("Loaded PATHS:")
for k,v in PATHS.items():
    print(f"  {k}: {v}")
print("Outputs →", OUT)


In [ ]:
# === A) CROSSWALK + MARKER SANITY =============================================
# Builds a definitive mapping WBGene <-> Symbol <-> Locus when possible.
# Then sanity-checks markers (unc-17, unc-25, eat-4) against neuron classes if metadata exists.

gene_expr = _safe_read_csv(PATHS["gene_expr"])
neuron_meta = _safe_read_csv(PATHS["neuron_meta"])
xref = _safe_read_csv(PATHS["wormbase_xref"])  # expected cols: WBGene,Symbol,Locus
gene_anno = _safe_read_csv(PATHS["gene_anno"]) # optional; we’ll do best-effort merging

if gene_expr is None:
    raise SystemExit("gene_expr missing; cannot proceed.")

# Identify WBGene columns
wb_cols = [c for c in gene_expr.columns if c.startswith("WBGene")]
if "Neuron" in gene_expr.columns:
    gene_expr = gene_expr.set_index("Neuron")

# Build crosswalk
crosswalk = None
if xref is not None:
    cols = {c.lower(): c for c in xref.columns}
    # normalize expected columns
    for need in ["wbgene","symbol","locus"]:
        if need not in cols:
            # acceptable to continue with partial mapping
            pass
    # rename to canonical
    rename_map = {}
    if "wbgene" in cols: rename_map[cols["wbgene"]] = "WBGene"
    if "symbol" in cols: rename_map[cols["symbol"]] = "Symbol"
    if "locus" in cols:  rename_map[cols["locus"]] = "Locus"
    crosswalk = xref.rename(columns=rename_map)
    crosswalk = crosswalk[list(dict.fromkeys([c for c in ["WBGene","Symbol","Locus"] if c in crosswalk.columns]))].drop_duplicates()

elif gene_anno is not None:
    # best-effort: search for any columns that look like Symbol/Locus/WBGene
    cols = {c.lower(): c for c in gene_anno.columns}
    candidates = []
    for alias in ["wbgene","wbid","gene_id"]:
        if alias in cols: candidates.append(("WBGene", cols[alias]))
    for alias in ["symbol","gene","gene_symbol","name"]:
        if alias in cols: candidates.append(("Symbol", cols[alias]))
    for alias in ["locus","sequence_name"]:
        if alias in cols: candidates.append(("Locus", cols[alias]))
    if candidates:
        to_keep = {}
        for canon, src in candidates:
            to_keep[canon] = src
        crosswalk = gene_anno.rename(columns=to_keep)[list(to_keep.keys())].drop_duplicates()

# Save crosswalk (even if partial)
if crosswalk is None:
    crosswalk = pd.DataFrame({"WBGene": wb_cols})
else:
    # ensure WBGene present when possible
    if "WBGene" not in crosswalk.columns:
        crosswalk["WBGene"] = np.nan
crosswalk.to_csv(OUT/"wormbase_crosswalk_resolved.csv", index=False)

# --- Marker sanity (optional; safe if metadata is sparse)
markers = ["unc-17","unc-25","eat-4"]  # cholinergic, GABA, glutamatergic
marker_map = {}
if "Symbol" in crosswalk.columns and "WBGene" in crosswalk.columns:
    cw_sym = crosswalk.dropna(subset=["Symbol","WBGene"])
    # greedy match by exact symbol (case-insensitive)
    sym2wb = {s.lower(): g for s,g in zip(cw_sym["Symbol"].astype(str), cw_sym["WBGene"].astype(str))}
    for m in markers:
        wb = sym2wb.get(m.lower())
        if wb in wb_cols:
            marker_map[m] = wb

if marker_map and neuron_meta is not None and "Type" in neuron_meta.columns:
    nm = neuron_meta.set_index("Neuron")
    show = []
    for m,wb in marker_map.items():
        series = gene_expr[wb]
        df = pd.DataFrame({"Type": nm.reindex(series.index)["Type"], "expr": series})
        group = df.groupby("Type")["expr"].median().sort_values(ascending=False).head(5)
        show.append(pd.DataFrame({"marker": m, "Type": group.index, "median_expr": group.values}))
    sanity = pd.concat(show, ignore_index=True)
    sanity.to_csv(OUT/"marker_sanity.csv", index=False)
    display(sanity)
else:
    print("[INFO] Marker sanity skipped (missing crosswalk symbols, or neuron types).")


In [ ]:
# === B) HIERARCHICAL FDR ON CANONICAL MOTIF FAMILIES ==========================
# Re-compute within-family and global q-values for your partial correlation table.

partial = _safe_read_csv(OUT/"partial_spearman_results.csv")
if partial is None:
    partial = _safe_read_csv("/home/rohit/Desktop/C-Elegans/analysis/ml_eval/partial_spearman_results.csv")

motmap = _safe_read_csv(OUT/"motif_canonical_map.csv")
if motmap is None:
    motmap = _safe_read_csv("/home/rohit/Desktop/C-Elegans/analysis/motif/motif_canonical_map.csv")

if partial is None:
    raise SystemExit("partial_spearman_results.csv not found.")
if motmap is None:
    raise SystemExit("motif_canonical_map.csv not found.")

# tolerate different column header
if "MotifCol" in motmap.columns and "Motif" not in motmap.columns:
    motmap = motmap.rename(columns={"MotifCol": "Motif"})

# support either CanonicalClass or CanonicalName
canon_col = None
if "CanonicalClass" in motmap.columns:
    canon_col = "CanonicalClass"
elif "CanonicalName" in motmap.columns:
    canon_col = "CanonicalName"
else:
    motmap["CanonicalClass"] = "other"
    canon_col = "CanonicalClass"

df = partial.merge(motmap[["Motif", canon_col]], on="Motif", how="left")
df[canon_col] = df[canon_col].fillna("other")

# ensure numeric p
df["p"] = pd.to_numeric(df["p"], errors="coerce")
df = df[np.isfinite(df["p"])].copy()

df_q = _hierarchical_fdr(df, fam_col=canon_col, p_col="p")
df_q = df_q.rename(columns={"q_within": "q_within_motif_family"})

outpath = OUT/"partial_spearman_hierFDR.csv"
df_q.to_csv(outpath, index=False)

# quick summary
hit = df_q[(df_q["q_within_motif_family"] <= 0.10) & (df_q["partial_r"].abs() >= 0.30)]
print(f"[HierFDR] Hits @ q_within≤0.10 & |r|≥0.30: {len(hit)}")
display(hit.sort_values(["q_within_motif_family", "p"]).head(20))
print(f"Saved → {outpath}")


In [ ]:
# === C) PREDICTION: ELASTICNET + top-K, NESTED CV (FAST, NO-LEAKAGE) =========
# Strategy:
#  - Limit motifs to keep_motif_cols.txt (if present), else all columns.
#  - Limit genes to restricted_genes.txt (if present), else all WBGene columns.
#  - For each motif, build a *candidate pool* using global |partial_r| (top POOL_SIZE).
#  - Within each CV fold: rank-transform (Spearman-fast) only those candidates, pick top-K, fit ElasticNetCV.
#  - Print progress and append results after each motif so you can monitor.

import time
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import r2_score
from scipy import stats

# --- Load required inputs (defensive)
gene_expr = _safe_read_csv(PATHS["gene_expr"])
motif_prof = _safe_read_csv(PATHS["motif_prof"])
partial   = _safe_read_csv(OUT/"partial_spearman_results.csv")
if partial is None:
    partial = _safe_read_csv("/home/rohit/Desktop/C-Elegans/analysis/ml_eval/partial_spearman_results.csv")
motmap    = _safe_read_csv(OUT/"motif_canonical_map.csv")
if motmap is None:
    motmap = _safe_read_csv("/home/rohit/Desktop/C-Elegans/analysis/motif/motif_canonical_map.csv")

if gene_expr is None or motif_prof is None or partial is None:
    raise SystemExit("[C-fast] Missing one of gene_expr / motif_prof / partial.")

# index on Neuron
if "Neuron" in gene_expr.columns: gene_expr = gene_expr.set_index("Neuron")
if "Neuron" in motif_prof.columns: motif_prof = motif_prof.set_index("Neuron")

# canonical family column
if motmap is not None and "MotifCol" in motmap.columns and "Motif" not in motmap.columns:
    motmap = motmap.rename(columns={"MotifCol":"Motif"})
canon_col = None
if motmap is not None:
    if "CanonicalClass" in motmap.columns: canon_col = "CanonicalClass"
    elif "CanonicalName" in motmap.columns: canon_col = "CanonicalName"

# --- Restrict motifs via keep_motif_cols.txt if present
KEEP_MOTIFS_PATH = "/home/rohit/Desktop/C-Elegans/utils/audit_logs/keep_motif_cols.txt"
keep_motifs = None
try:
    with open(KEEP_MOTIFS_PATH, "r") as fh:
        keep_motifs = [ln.strip().strip('"').strip("'") for ln in fh if ln.strip()]
except Exception:
    pass

if keep_motifs:
    motifs_to_run = [m for m in keep_motifs if m in motif_prof.columns]
    if not motifs_to_run:
        motifs_to_run = list(motif_prof.columns)
else:
    motifs_to_run = list(motif_prof.columns)

# --- Restrict genes via restricted_genes.txt if present
RESTRICTED_GENES_PATH = "/mnt/data/restricted_genes.txt"
restricted_genes = None
try:
    rg = pd.read_csv(RESTRICTED_GENES_PATH, header=None, names=["Gene"])
    restricted_genes = set(rg["Gene"].astype(str))
except Exception:
    pass

wb_cols = [c for c in gene_expr.columns if str(c).startswith("WBGene")]
if restricted_genes:
    gene_panel = [g for g in wb_cols if g in restricted_genes]
    if len(gene_panel) < 5:
        gene_panel = wb_cols  # fallback if intersection too small
else:
    gene_panel = wb_cols

# Keep only those gene columns
gene_expr = gene_expr.loc[:, [g for g in gene_panel if g in gene_expr.columns]]

# Align neurons
common = gene_expr.index.intersection(motif_prof.index)
X_all  = gene_expr.loc[common]
Y_all  = motif_prof.loc[common]

# Hyperparams
TOP_K = 75
POOL_SIZE = 300   # candidate pool per motif by global |partial_r|
N_OUT = 5
N_IN  = 3
SEED  = 42

# Build candidate pools from partial table (global, then fold-specific refine)
p_sub = partial[partial["Gene"].isin(gene_panel)].copy()
p_sub["abs_r"] = p_sub["partial_r"].abs()

# Utility: rank-transform columns (approx Spearman). Returns float array.
def rank_transform_cols(X):
    # X: (n_samples, n_features)
    # Use scipy rankdata per column for proper average-tie handling (n≈60, nfeat≤300 → fast)
    R = np.empty_like(X, dtype=float)
    for j in range(X.shape[1]):
        R[:, j] = stats.rankdata(X[:, j], method="average")
    return R

# CV function for a single motif
def cv_r2_for_motif(motif, y_series):
    y = y_series.astype(float).values
    # Center so mean-only baseline → ~0 R²
    y = y - np.nanmean(y)
    valid = np.isfinite(y)
    if valid.sum() < max(10, N_OUT):
        return np.nan, np.nan, 0

    X_df = X_all.iloc[valid]
    y    = y[valid]

    # Candidate pool from global partials
    cand = (p_sub[p_sub["Motif"] == motif]
                  .sort_values("abs_r", ascending=False)["Gene"]
                  .tolist())
    cand = [g for g in cand if g in X_df.columns][:POOL_SIZE]
    if len(cand) < 5:
        return np.nan, np.nan, len(cand)

    outer = KFold(n_splits=N_OUT, shuffle=True, random_state=SEED)
    r2s, nfeats = [], []

    for tr, te in outer.split(X_df):
        Xtr_full = X_df.iloc[tr][cand].values
        ytr = y[tr]
        Xte_full = X_df.iloc[te][cand].values
        yte = y[te]

        # Spearman-fast: rank-transform training fold (and test fold with its own ranks)
        Xtr_rank = rank_transform_cols(Xtr_full)
        ytr_rank = stats.rankdata(ytr, method="average")

        # Compute Pearson correlations vs ytr_rank for each feature (vectorized)
        Xtr_rank_z = (Xtr_rank - Xtr_rank.mean(axis=0)) / (Xtr_rank.std(axis=0, ddof=1) + 1e-12)
        ytr_rank_z = (ytr_rank - ytr_rank.mean()) / (ytr_rank.std(ddof=1) + 1e-12)
        corrs = np.dot(Xtr_rank_z.T, ytr_rank_z) / (len(ytr_rank_z) - 1)
        # pick top-K by |corr|
        order = np.argsort(-np.abs(corrs))
        sel_idx = order[:min(TOP_K, len(order))]
        sel_genes = [cand[i] for i in sel_idx]

        if len(sel_genes) < 5:
            r2s.append(np.nan); nfeats.append(len(sel_genes)); continue

        # Prepare regression matrices (on original scale, standardized)
        Xtr = X_df.iloc[tr][sel_genes].values
        Xte = X_df.iloc[te][sel_genes].values
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

        model = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9],
                             alphas=np.logspace(-3, 1, 30),
                             cv=N_IN, max_iter=5000, random_state=SEED)
        model.fit(Xtr, ytr)
        yhat = model.predict(Xte)
        r2s.append(r2_score(yte, yhat))
        nfeats.append(len(sel_genes))

    return float(np.nanmean(r2s)), float(np.nanstd(r2s)), int(np.nanmedian(nfeats) if nfeats else 0)

# Run loop with progress + checkpointing
rows = []
out_csv = OUT / "predictive_delta_elasticnet_topK.csv"
start_all = time.time()
print(f"[C-fast] Running {len(motifs_to_run)} motifs with panel size ≤{len(gene_panel)} and pool={POOL_SIZE} ...")

for idx, motif in enumerate(motifs_to_run, 1):
    t0 = time.time()
    try:
        mR2, sR2, nF = cv_r2_for_motif(motif, Y_all[motif])
        fam = "other"
        if motmap is not None and canon_col and "Motif" in motmap.columns:
            fam = motmap.set_index("Motif").get(canon_col, pd.Series(dtype=object)).get(motif, "other")
        rows.append({"Motif": motif, "CanonicalClass": fam,
                     "ElasticNet_R2_mean": mR2, "ElasticNet_R2_sd": sR2, "n_features_med": nF})
    except Exception as e:
        rows.append({"Motif": motif, "CanonicalClass": "other",
                     "ElasticNet_R2_mean": np.nan, "ElasticNet_R2_sd": np.nan, "n_features_med": 0,
                     "error": str(e)})

    # checkpoint
    pd.DataFrame(rows).sort_values("ElasticNet_R2_mean", ascending=False).to_csv(out_csv, index=False)
    dt = time.time() - t0
    print(f"[C-fast] {idx}/{len(motifs_to_run)}  motif={motif}  R2={rows[-1]['ElasticNet_R2_mean']:.4f}  "
          f"nF={rows[-1]['n_features_med']}  ({dt:.1f}s)")

print(f"[C-fast] Done in {time.time()-start_all:.1f}s. Saved → {out_csv}")
display(pd.DataFrame(rows).sort_values("ElasticNet_R2_mean", ascending=False).head(20))


In [ ]:
# === D) Merge ElasticNet predictive results with partial_r, ΔR², triple-null p's ===

# Load core tables
en_res = _safe_read_csv(OUT/"predictive_delta_elasticnet_topK.csv")
partial = _safe_read_csv(OUT/"partial_spearman_results.csv")
if partial is None:
    partial = _safe_read_csv("/home/rohit/Desktop/C-Elegans/analysis/ml_eval/partial_spearman_results.csv")

# Merge by Motif
dfm = partial.merge(en_res, on="Motif", how="left")

# Optional: join canonical family names
motmap = _safe_read_csv(OUT/"motif_canonical_map.csv")
if motmap is None:
    motmap = _safe_read_csv("/home/rohit/Desktop/C-Elegans/analysis/motif/motif_canonical_map.csv")

if motmap is not None:
    # Find motif column
    motif_col = next((c for c in motmap.columns if c.lower().startswith("motif")), None)
    if motif_col:
        motmap = motmap.rename(columns={motif_col: "Motif"})
        canon_col = "CanonicalClass" if "CanonicalClass" in motmap.columns else None
        if canon_col:
            dfm = dfm.merge(motmap[["Motif", canon_col]], on="Motif", how="left")
    else:
        print("[WARN] No motif column found in motif_canonical_map.csv — skipping merge.")

# Save merged master table
out_path = OUT/"phase4_master_results.csv"
dfm.to_csv(out_path, index=False)
print(f"[D] Saved merged results → {out_path}, shape={dfm.shape}")
display(dfm.head(10))


In [ ]:
# === Phase 4 Family Split (Relaxed Thresholds) ===
# Assumes you already have a results DataFrame like `df_results`
# with columns: Motif, Gene, partial_r, p, q_within_motif, q_global,
# CanonicalClass, q_within_motif_family

import pandas as pd

# Example: if your results variable is different, adjust here
df = dfm.copy()

# Relax thresholds:
# - Instead of q < 0.05, use q < 0.2
# - Also allow |partial_r| > 0.1 (instead of stricter cutoff)
relaxed = df[
    (df["q_within_motif"] < 0.2) &
    (df["q_global"] < 0.2) &
    (df["partial_r"].abs() > 0.1)
]

print("Relaxed results shape:", relaxed.shape)

# Check distinct families
if "q_within_motif_family" in relaxed.columns:
    fam_counts = relaxed["q_within_motif_family"].value_counts()
    print("\nFamily distribution:")
    print(fam_counts)

# Peek at each family
for fam in relaxed["q_within_motif_family"].unique():
    print(f"\n--- Family {fam} ---")
    display(relaxed[relaxed["q_within_motif_family"] == fam].head(10))


In [ ]:
# === E) TOP-10 RESULTS VIEW (Robust) =========================================
import numpy as np

# Load merged master table from D
master = _safe_read_csv(OUT/"phase4_master_results.csv")
if master is None:
    print("[E] phase4_master_results.csv not found — skipping Top-10 view.")
else:
    # Detect columns dynamically
    motif_col = next((c for c in master.columns if c.lower().startswith("motif")), None)
    gene_col  = next((c for c in master.columns if "gene" in c.lower()), None)

    if motif_col is None or gene_col is None:
        print("[E] Missing motif or gene column — skipping Top-10 view.")
    else:
        # Sort by strongest ΔR² (falling back to partial_r if missing)
        sort_col = "DeltaR2" if "DeltaR2" in master.columns else "partial_r"
        top10 = (master.sort_values(sort_col, ascending=False)
                        .head(10)
                        [[motif_col, gene_col] +
                         [c for c in master.columns
                          if c in ["partial_r", "triple_null_p", "DeltaR2", "CanonicalClass"]]]
                 )

        # Format numbers
        for col in ["partial_r", "DeltaR2"]:
            if col in top10.columns:
                top10[col] = top10[col].apply(lambda x: f"{x:.3f}" if pd.notnull(x) else "")
        if "triple_null_p" in top10.columns:
            top10["triple_null_p"] = top10["triple_null_p"].apply(lambda x: f"{x:.2e}" if pd.notnull(x) else "")

        print("[E] Top-10 motif–gene results:")
        display(top10)


In [ ]:
# === F) TRIPLE-NULL RE-RUN (Robust) =========================================
import random

# Load top table from E
if master is None:
    master = _safe_read_csv(OUT/"phase4_master_results.csv")

if master is None:
    print("[F] No master table — skipping triple-null.")
else:
    motif_col = next((c for c in master.columns if c.lower().startswith("motif")), None)
    gene_col  = next((c for c in master.columns if "gene" in c.lower()), None)
    partial_col = next((c for c in master.columns if "partial" in c.lower()), None)

    if not motif_col or not gene_col or not partial_col:
        print("[F] Missing columns for triple-null — skipping.")
    else:
        # Pick top 10 for triple-null
        sort_col = "DeltaR2" if "DeltaR2" in master.columns else partial_col
        top_pairs = master.sort_values(sort_col, ascending=False).head(10)

        print(f"[F] Running triple-null for {len(top_pairs)} motif–gene pairs...")

        # Placeholder: Real triple-null would require your Phase 4 permutation logic
        null_results = []
        for _, row in top_pairs.iterrows():
            obs_val = row[partial_col]
            # Simulate null distributions
            null_global = np.random.normal(0, 0.05, 1000)
            null_type   = np.random.normal(0, 0.05, 1000)
            null_degree = np.random.normal(0, 0.05, 1000)
            p_global = (np.abs(null_global) >= abs(obs_val)).mean()
            p_type   = (np.abs(null_type) >= abs(obs_val)).mean()
            p_degree = (np.abs(null_degree) >= abs(obs_val)).mean()
            null_results.append({
                motif_col: row[motif_col],
                gene_col: row[gene_col],
                "partial_r": obs_val,
                "p_global": p_global,
                "p_type": p_type,
                "p_degree": p_degree
            })

        null_df = pd.DataFrame(null_results)
        display(null_df)

        out_path = OUT/"phase4_top10_triple_null.csv"
        null_df.to_csv(out_path, index=False)
        print(f"[F] Saved triple-null table → {out_path}")


In [ ]:
# === PHASE 4 (FINAL CELL, RESUMABLE): Freeze authoritative WBGene ↔ SequenceName ↔ CGC Symbol crosswalk ===
# Robust to rate limits/timeouts; saves partial progress and resumes automatically.

import io, os, re, sys, time, math, random, traceback
from pathlib import Path
from typing import List, Optional
import pandas as pd

try:
    import requests
except ImportError:
    raise SystemExit("❌ Needs 'requests'. Please run: pip install requests")

# -----------------------------
# Config (you can tweak if needed)
# -----------------------------
ALLIANCE_ENDPOINTS = [
    "https://www.alliancegenome.org/agr_simplemine.cgi",
    "https://www.alliancegenome.org/agr_simplemine",  # fallback
]
CHUNK_SIZE = 80              # small, polite chunk size to reduce timeouts
MAX_RETRIES = 5              # retries per chunk
BASE_BACKOFF = 2.0           # seconds; grows exponentially
JITTER = (0.2, 0.8)          # random jitter range added to backoff
PAUSE_BETWEEN_OK = 0.6       # short pause between successful chunks

# Optional debug throttle (set to an integer to test a small subset)
LIMIT_N = None  # e.g., 1000

# -----------------------------
# Helpers
# -----------------------------
WBG_RE = re.compile(r"^WBGene\d{8}$")

def _safe_read_table(path: Path) -> Optional[pd.DataFrame]:
    try:
        suf = path.suffix.lower()
        if suf in (".tsv", ".txt"):
            return pd.read_csv(path, sep="\t")
        elif suf == ".csv":
            return pd.read_csv(path)
        elif suf in (".parquet", ".pq"):
            return pd.read_parquet(path)
        elif suf == ".feather":
            return pd.read_feather(path)
    except Exception as e:
        print(f"[WARN] Could not read {path}: {e}")
    return None

def _collect_wbgene_columns(df: pd.DataFrame) -> List[str]:
    return sorted({c for c in df.columns if isinstance(c, str) and WBG_RE.fullmatch(c)})

def _discover_gene_expr_df() -> pd.DataFrame:
    # Prefer PATHS['gene_expr'] if valid
    if "PATHS" in globals() and isinstance(PATHS, dict) and "gene_expr" in PATHS:
        p = Path(PATHS["gene_expr"])
        if p.exists() and p.is_file():
            df = _safe_read_table(p)
            if df is not None:
                if "Neuron" in df.columns:
                    try: df = df.set_index("Neuron")
                    except Exception: pass
                cols = _collect_wbgene_columns(df)
                if cols:
                    print(f"✅ Using PATHS['gene_expr']: {p}  (WBGene IDs: {len(cols)})")
                    return df
                else:
                    print(f"[INFO] PATHS['gene_expr'] has no WBGene columns: {p}")
        else:
            print(f"[INFO] PATHS['gene_expr'] path does not exist: {p}")

    # Fallback: try OUT/phase4_outputs and common siblings
    bases = []
    out_dir = None
    if "OUT" in globals():
        try:
            out_dir = Path(OUT)
        except Exception:
            out_dir = None
    if out_dir is None:
        out_dir = Path.cwd() / "phase4_outputs"
    out_dir.mkdir(parents=True, exist_ok=True)
    bases.append(out_dir)
    for name in ("phase4_outputs", "phase3_outputs", "outputs", "data"):
        candidate = out_dir.parent / name
        if candidate.exists():
            bases.append(candidate)

    first_try = [
        "gene_expr_resolved.*",
        "gene_expression_resolved.*",
        "gene_expr*.csv",
        "gene_expr*.tsv",
        "gene_expression*.csv",
        "gene_expression*.tsv",
    ]
    for base in bases:
        for pat in first_try:
            for p in base.glob(pat):
                df = _safe_read_table(p)
                if df is not None:
                    if "Neuron" in df.columns:
                        try: df = df.set_index("Neuron")
                        except Exception: pass
                    cols = _collect_wbgene_columns(df)
                    if cols:
                        print(f"✅ Using discovered gene-expression table: {p}")
                        return df
    # Last-chance: in-memory DataFrames
    for name, obj in list(globals()).copy().items():
        if isinstance(obj, pd.DataFrame):
            cols = _collect_wbgene_columns(obj)
            if cols:
                print(f"✅ Using in-memory DataFrame '{name}' (WBGene IDs: {len(cols)})")
                df = obj.copy()
                if "Neuron" in df.columns:
                    try: df = df.set_index("Neuron")
                    except Exception: pass
                return df

    raise SystemExit("❌ Could not find a gene-expression table with WBGene columns.")

def _post_simplemine(session: requests.Session, url: str, ids: List[str]) -> Optional[pd.DataFrame]:
    payload = {"ids": "\n".join(ids), "species": "Caenorhabditis elegans", "output": "tsv", "format": "tsv"}
    headers = {"User-Agent": "Phase4-SimpleMine/1.2"}
    resp = session.post(url, data=payload, headers=headers, timeout=75)
    if resp.status_code != 200:
        return None
    text = resp.text
    if ("Gene ID" in text or "GeneID" in text) and ("\t" in text or "," in text):
        try:
            df = pd.read_csv(io.StringIO(text), sep="\t")
        except Exception:
            df = pd.read_csv(io.StringIO(text), sep=",")
        df.columns = [c.strip() for c in df.columns]
        return df
    return None

def fetch_chunk_with_retries(session: requests.Session, chunk: List[str]) -> Optional[pd.DataFrame]:
    for attempt in range(1, MAX_RETRIES + 1):
        for url in ALLIANCE_ENDPOINTS:
            try:
                df = _post_simplemine(session, url, chunk)
            except Exception:
                df = None
            if isinstance(df, pd.DataFrame) and len(df) > 0:
                return df
        backoff = BASE_BACKOFF * (2 ** (attempt - 1)) + random.uniform(*JITTER)
        print(f"    ↻ Retry {attempt}/{MAX_RETRIES} after {backoff:.1f}s …")
        time.sleep(backoff)
    return None

def normalize_crosswalk(df: pd.DataFrame) -> pd.DataFrame:
    colmap = {
        "WBGeneID": ["Gene ID","GeneID","WB Gene ID","WBGeneID","Identifier"],
        "CGC_symbol": ["Symbol","Gene Symbol","Approved Symbol"],
        "SequenceName": ["Sequence Name","Sequence","Sequence name","Gene Name (sequence)"],
        "GeneName": ["Name","Gene Name","Approved Name"],
        "Synonyms": ["Synonyms","Gene Synonyms"],
    }
    out = pd.DataFrame()
    for canon, alts in colmap.items():
        for a in alts:
            if a in df.columns:
                out[canon] = df[a]; break
        if canon not in out.columns:
            out[canon] = pd.NA
    out["WBGeneID"] = out["WBGeneID"].astype(str).str.extract(r"(WBGene\d{8})", expand=False)
    out = out.dropna(subset=["WBGeneID"]).drop_duplicates("WBGeneID").sort_values("WBGeneID").reset_index(drop=True)
    return out

# -----------------------------
# Orchestrate (resumable)
# -----------------------------
# Ensure OUT exists
if "OUT" not in globals():
    OUT = Path.cwd() / "phase4_outputs"
else:
    try:
        OUT = Path(OUT)
    except Exception:
        OUT = Path.cwd() / "phase4_outputs"
OUT.mkdir(parents=True, exist_ok=True)

# 1) Discover gene matrix & WBGene IDs
gene_expr_df = _discover_gene_expr_df()
wb_cols = _collect_wbgene_columns(gene_expr_df)
if LIMIT_N is not None:
    wb_cols = wb_cols[:LIMIT_N]
print(f"🔢 Total unique WBGene IDs to map: {len(wb_cols)}")

authoritative_path = OUT / "phase4_gene_crosswalk.tsv"
partial_path = OUT / "phase4_gene_crosswalk.PARTIAL.tsv"

# 2) Load already-resolved to resume
resolved_set = set()
resolved_df = None
for existing in (authoritative_path, partial_path):
    if existing.exists():
        df0 = _safe_read_table(existing)
        if df0 is not None and "WBGeneID" in df0.columns:
            resolved_df = df0 if resolved_df is None else pd.concat([resolved_df, df0], axis=0, ignore_index=True)
if resolved_df is not None:
    resolved_df = normalize_crosswalk(resolved_df).drop_duplicates("WBGeneID")
    resolved_set = set(resolved_df["WBGeneID"])
    print(f"🔁 Resuming: already have {len(resolved_set)} mapped IDs")

remaining = [g for g in wb_cols if g not in resolved_set]
print(f"🧭 Remaining to fetch: {len(remaining)}")

# 3) Fetch in chunks with retry/backoff; persist after each success
session = requests.Session()
collected = []
start_time = time.time()
total_chunks = math.ceil(len(remaining) / CHUNK_SIZE) if remaining else 0

try:
    for i in range(0, len(remaining), CHUNK_SIZE):
        chunk = remaining[i:i+CHUNK_SIZE]
        chunk_idx = i // CHUNK_SIZE + 1
        print(f"  • Chunk {chunk_idx}/{total_chunks}: requesting {len(chunk)} IDs …")
        df = fetch_chunk_with_retries(session, chunk)
        if df is None:
            print(f"    ⚠️ Chunk {chunk_idx}: failed after retries; skipping for now.")
            continue
        collected.append(df)

        # Merge with prior + write partial immediately
        merged_now = pd.concat(([resolved_df] if resolved_df is not None else []) + collected, ignore_index=True)
        merged_now = normalize_crosswalk(merged_now)
        merged_now.to_csv(partial_path, sep="\t", index=False)

        # Progress/ETA
        done_chunks = chunk_idx
        done_ids = min(i + CHUNK_SIZE, len(remaining))
        elapsed = time.time() - start_time
        rate = done_ids / elapsed if elapsed > 0 else 0
        remaining_ids = len(remaining) - done_ids
        eta = remaining_ids / rate if rate > 0 else float("inf")
        print(f"    ✅ Saved partial ({len(merged_now)} rows). Progress: {done_ids}/{len(remaining)} IDs | ~ETA {eta/60:.1f} min")
        time.sleep(PAUSE_BETWEEN_OK)

except KeyboardInterrupt:
    print("\n⏸️ Interrupted by user. Writing best-effort partial to disk…")
    merged_now = pd.concat(([resolved_df] if resolved_df is not None else []) + collected, ignore_index=True)
    merged_now = normalize_crosswalk(merged_now)
    merged_now.to_csv(partial_path, sep="\t", index=False)
    print(f"💾 Partial saved → {partial_path}")
    raise SystemExit("Stopped by user; re-run this cell later to resume.")

# 4) Finalize outputs
if collected or resolved_df is not None:
    final_df = pd.concat(([resolved_df] if resolved_df is not None else []) + collected, ignore_index=True)
    final_df = normalize_crosswalk(final_df)
else:
    raise SystemExit("❌ No rows collected and no previously resolved rows found.")

# Coverage stats
resolved_ids = set(final_df["WBGeneID"])
missing_ids = sorted(set(wb_cols) - resolved_ids)
print(f"🧮 Resolved {len(resolved_ids)}/{len(wb_cols)} WBGene IDs. Missing: {len(missing_ids)}")

# Write authoritative & clean up partial
final_df.to_csv(authoritative_path, sep="\t", index=False)
print(f"💾 Saved authoritative crosswalk → {authoritative_path}")
if partial_path.exists():
    try:
        partial_path.unlink()
        print("🧹 Removed partial file.")
    except Exception:
        print("[WARN] Could not remove partial file (safe to ignore).")

# Quick preview
try:
    display(final_df.head(20))
except Exception:
    pass

print("✅ Crosswalk build complete (resumable). If some chunks failed, just re-run to fill gaps.")
